# RoPE — Solution

Reference implementation for `03_rope_exercise.ipynb`.

## Setup

In [1]:
import torch

## Solution 1 — Frequencies

In [2]:
def get_rope_frequencies(d_head, base=10000.0):
    return 1 / torch.pow(base, torch.arange(0, d_head, 2) / d_head)

## Solution 2 — Single 4D rotation (worked example)

Rotate `q = [1.0, 0.5, 0.8, -0.3]` at `pos = 1` with `d_head = 4`. Build the block-diagonal rotation matrix and apply it — should match the worked example in the blog post.

In [3]:
pos = 1
d_head = 4   # 4 dims, 2 pairs
freq = get_rope_frequencies(d_head)

q = torch.tensor([1.0, 0.5, 0.8, -0.3])
cos = torch.cos(freq * pos)
sin = torch.sin(freq * pos)

block_diag = [
    torch.tensor([[cos[i], -sin[i]],
                  [sin[i],  cos[i]]])
    for i in range(d_head // 2)
]
rot_matrix = torch.block_diag(*block_diag)

q_rot = rot_matrix @ q
q_rot

tensor([ 0.1196,  1.1116,  0.8030, -0.2920])

## Solution 3 — Batched RoPE

In [4]:
def apply_rope(x, positions):
    """Apply RoPE to a Q or K tensor.

    Args:
        x:         (batch, seq_len, n_heads, d_head)
        positions: (seq_len,) integer positions

    Returns:
        Same shape as x, with each dim pair rotated.
    """
    x_rotated = torch.zeros(*x.shape)
    frequencies = get_rope_frequencies(x.shape[-1])         # (d_head//2,)
    angles = positions[:, None] * frequencies[None, :]      # (seq_len, d_head//2)

    cos = torch.cos(angles).unsqueeze(1)                    # (seq_len, 1, d_head//2)
    sin = torch.sin(angles).unsqueeze(1)

    x_rotated[..., 0::2] = x[..., 0::2] * cos - x[..., 1::2] * sin
    x_rotated[..., 1::2] = x[..., 0::2] * sin + x[..., 1::2] * cos
    return x_rotated

In [5]:
# Quick shape check
out = apply_rope(torch.randn(2, 10, 4, 8), torch.arange(10))
out.shape

torch.Size([2, 10, 4, 8])

**Implementation choice: pair-adjacent vs pair-half.** The GPT-NeoX vs LLaMA convention difference — some implementations rotate pairs `(0,1), (2,3), ...` while others rotate `(0, d/2), (1, d/2+1), ...`. The math is equivalent, but be careful when porting weights between models.

## Solution 4 — Same-offset, same-score demonstration

In [6]:
def rope_at(x, pos):
    x_4d = x.view(1, 1, 1, -1)             # (1, 1, 1, d_head)
    positions = torch.tensor([pos])        # (1,)
    return apply_rope(x_4d, positions).view(-1)


def score(i, j):
    q_i = rope_at(q_base, i)
    k_j = rope_at(k_base, j)
    return (q_i * k_j).sum().item()

In [7]:
d_head = 64
q_base = torch.randn(d_head)
k_base = torch.randn(d_head)

In [8]:
# Experiment 1: same offset, very different absolute positions -> same score
print('Same offset (j - i = 4), different absolute positions:')
s1 = score(3, 7)
s2 = score(100, 104)
s3 = score(1000, 1004)
s4 = score(5000, 5004)
print(f'  score(3, 7)       = {s1:+.6f}')
print(f'  score(100, 104)   = {s2:+.6f}')
print(f'  score(1000, 1004) = {s3:+.6f}')
print(f'  score(5000, 5004) = {s4:+.6f}')

assert abs(s1 - s2) < 1e-3, f'score should be position-invariant: {s1} vs {s2}'
assert abs(s1 - s3) < 1e-3, f'score should be position-invariant: {s1} vs {s3}'

# Experiment 2: different offsets -> different scores
print('\nDifferent offsets, same starting position:')
print(f'  score(3, 7)   (offset 4) = {score(3, 7):+.6f}')
print(f'  score(3, 9)   (offset 6) = {score(3, 9):+.6f}')
print(f'  score(3, 11)  (offset 8) = {score(3, 11):+.6f}')

assert score(3, 7) != score(3, 9), 'different offsets should give different scores'

print('\nRoPE attention scores depend ONLY on relative position (j - i).')

Same offset (j - i = 4), different absolute positions:
  score(3, 7)       = -7.352024
  score(100, 104)   = -7.352013
  score(1000, 1004) = -7.352046
  score(5000, 5004) = -7.351250

Different offsets, same starting position:
  score(3, 7)   (offset 4) = -7.352024
  score(3, 9)   (offset 6) = -5.925401
  score(3, 11)  (offset 8) = -7.513608

RoPE attention scores depend ONLY on relative position (j - i).
